# Práctica 03 — El Perceptrón
**Asignatura:** Inteligencia Artificial  
**Estudiante:** Daniel Lara  
Implementación manual en Python, sin librerías de machine learning.

## Fase 1. Exploración del dataset — Actividad 1.a

In [ ]:
X = [[1,7],[1,3],[3,4],[4,6],[4,2],[6,6],[6,2],[6,0],[7,4],[9,3],[9,1],[11,1],[12,3],[16,2],
     [4,8],[6,8],[8,9],[9,7],[10,11],[12,9],[12,6],[14,7],[16,4]]
y = [-1]*14 + [1]*9
print("Total de instancias:", len(X))
print("Variables de entrada: x1, x2")
print("Número de clases:", len(set(y)))
print("Clase -1:", y.count(-1), "| Clase +1:", y.count(1))

**Pregunta guía 1.a.** Cada fila de `X` representa una instancia con dos características numéricas. Cada valor de `y` identifica la clase de la instancia correspondiente. Es clasificación binaria porque existen exactamente dos etiquetas posibles: -1 y +1.

### Actividad 1.b — Separación visual de clases

In [ ]:
import matplotlib.pyplot as plt
import random

def plot_points(X, y, title="Datos originales"):
    for clase, color, marcador in [(-1,"blue","o"),(1,"red","+")]:
        puntos = [p for p, etiqueta in zip(X,y) if etiqueta == clase]
        plt.scatter([p[0] for p in puntos], [p[1] for p in puntos], c=color, marker=marcador, label=f"Clase {clase}")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title(title)
    plt.grid(alpha=.3); plt.legend(); plt.show()
plot_points(X,y)

**Pregunta guía 1.b.** Sí, visualmente parece posible separar las clases mediante una recta inclinada: los puntos positivos se concentran principalmente por encima de los negativos, aunque la separación depende de ambas coordenadas.

## Fase 2. Fundamento del perceptrón — Actividades 2.a, 2.b y 2.c

In [ ]:
random.seed(42)  # Semilla para reproducir el experimento

def initialize_weights():
    return [random.uniform(-0.5,0.5) for _ in range(3)]

def activation(z):
    return 1 if z >= 0 else -1

def perceptron_compute(x,w):
    z = w[0] + w[1]*x[0] + w[2]*x[1]
    return activation(z)

w_demo = initialize_weights()
print("Pesos de prueba:", w_demo)
for x, real in zip(X[:3],y[:3]):
    print("Instancia:",x,"| Real:",real,"| Predicción:",perceptron_compute(x,w_demo))

**Pregunta 2.a.** Hay tres pesos porque dos multiplican las entradas y el tercero es el sesgo (`bias`). El sesgo desplaza la recta sin obligarla a pasar por el origen.

**Pregunta 2.b.** Las etiquetas -1 y +1 permiten representar simétricamente las dos clases y usar directamente el signo de la suma ponderada.

**Pregunta 2.c.** `z` es un valor numérico continuo; la activación transforma ese valor en una etiqueta binaria para poder clasificar.

## Fase 3. Frontera de decisión — Actividades 3.a y 3.b

In [ ]:
def perceptron_plot(X,y,w,title):
    plt.figure(figsize=(7,5))
    for clase, color, marcador in [(-1,"blue","o"),(1,"red","+")]:
        puntos=[p for p, etiqueta in zip(X,y) if etiqueta==clase]
        plt.scatter([p[0] for p in puntos],[p[1] for p in puntos],c=color,marker=marcador,label=f"Clase {clase}")
    if abs(w[2]) > 1e-12:
        xx=[-1,18]
        yy=[-(w[0]+w[1]*v)/w[2] for v in xx]
        plt.plot(xx,yy,"k-",label="Frontera de decisión")
    elif abs(w[1]) > 1e-12:
        plt.axvline(-w[0]/w[1],color="black",label="Frontera de decisión")
    else:
        print("Los pesos no definen una recta de decisión válida.")
    plt.xlim(-1,18); plt.ylim(-2,13)
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title(title)
    plt.legend(); plt.grid(alpha=.3); plt.show()

w_initial=initialize_weights()
perceptron_plot(X,y,w_initial,"Frontera inicial del perceptrón")
print("Pesos iniciales:",w_initial)
print("Aciertos iniciales:",sum(perceptron_compute(x,w_initial)==t for x,t in zip(X,y)),"de",len(X))

**Pregunta 3.a.** La recta representa los puntos donde la suma ponderada es cero; separa las regiones predichas como -1 y +1. Con pesos aleatorios, su posición y orientación son arbitrarias.

**Pregunta 3.b.** No se espera que la frontera inicial clasifique correctamente todos los puntos, porque todavía no ha aprendido a partir de los errores del dataset. El conteo anterior permite comprobarlo.

## Fase 4. Entrenamiento — Actividades 4.a, 4.b y 4.c

In [ ]:
def perceptron_learning(X,y,w,lr,max_epochs):
    w=w.copy()
    W_values=[w.copy()]  # Estado inicial
    E_values=[]
    updates=0
    for epoch in range(max_epochs):
        errors=0
        for x, real in zip(X,y):
            pred=perceptron_compute(x,w)
            if pred != real:
                errors+=1
                delta=lr*(real-pred)
                w[0]+=delta
                w[1]+=delta*x[0]
                w[2]+=delta*x[1]
                updates+=1
        E_values.append(errors/len(X))
        W_values.append(w.copy())
        if errors==0:
            break
    return w, len(E_values), updates, W_values, E_values
print("Función de aprendizaje implementada.")

**Pregunta 4.a.** El máximo de épocas evita un ciclo indefinido. Si los datos no fueran linealmente separables, el perceptrón podría seguir modificando los pesos sin alcanzar cero errores.

**Pregunta 4.b.** Una tasa de error cero significa que durante esa época ninguna instancia fue clasificada incorrectamente.

**Pregunta 4.c.** Se guardan copias para conservar el historial real de cada época. Si todas las entradas apuntaran al mismo objeto mutable, la evolución podría mostrar repetidamente los últimos pesos.

## Fase 5. Entrenamiento con lr = 0.5 — Actividades 5.a y 5.b

In [ ]:
w=initialize_weights()
print("Pesos iniciales:",w)
final_w,epochs,updates,W_values,E_values=perceptron_learning(X,y,w,lr=0.5,max_epochs=1000)
print("Pesos finales:",final_w)
print("Épocas:",epochs)
print("Actualizaciones:",updates)
print("Error final:",E_values[-1])
print("Errores finales al evaluar todos los puntos:",sum(perceptron_compute(x,final_w)!=t for x,t in zip(X,y)))
perceptron_plot(X,y,final_w,"Frontera final con lr = 0.5")

**Pregunta 5.a.** El modelo converge si `E_values[-1]` es cero y todas las instancias quedan correctamente clasificadas. El resultado numérico y la gráfica anteriores permiten verificarlo.

**Pregunta 5.b.** Si converge, la frontera final separa las dos clases. A diferencia de la frontera inicial aleatoria, la final refleja los ajustes realizados después de observar los errores.

## Fase 6. Nuevas instancias — Actividades 6.a y 6.b

In [ ]:
X_test=[[5,5],[10,6],[6,10]]
y_test=[perceptron_compute(x,final_w) for x in X_test]
print(f"{'Instancia':<18} Salida predicha")
for x,pred in zip(X_test,y_test):
    print(f"{str(x):<18} {pred:+d}")
X_extended=[p.copy() for p in X]+[p.copy() for p in X_test]
y_extended=y.copy()+y_test.copy()
print("Tamaño del dataset extendido:",len(X_extended))
perceptron_plot(X_extended,y_extended,final_w,"Dataset extendido y frontera aprendida")

**Pregunta 6.a.** Las predicciones son coherentes con la frontera aprendida porque la función de activación asigna cada punto al lado correspondiente de la recta.

**Pregunta 6.b.** Las nuevas instancias se sitúan en las regiones indicadas por sus predicciones. Los puntos cercanos a la recta pueden resultar discutibles, porque un pequeño cambio de coordenadas podría modificar su clase. Las etiquetas nuevas son predicciones, no valores reales verificados.

## Fase 7. Evolución de la frontera — Actividades 7.a y 7.b

In [ ]:
def plot_six_states(X,y,W_values,title):
    n=len(W_values)
    if n<6:
        indices=list(range(n))
    else:
        indices=[round((n-1)*p) for p in [0,.2,.4,.6,.8,1]]
    labels=[f"{round(100*i/(n-1)) if n>1 else 0}%" for i in indices]
    for i,lab in zip(indices,labels):
        perceptron_plot(X,y,W_values[i],f"{title} — {lab} (estado {i})")
plot_six_states(X,y,W_values,"Evolución con lr = 0.5")

**Pregunta 7.a.** La frontera cambia al modificarse los pesos: puede rotar o desplazarse y los cambios no tienen por qué ser uniformes, ya que solo hay actualizaciones ante errores.

**Pregunta 7.b.** La pendiente de la recta es `-w1/w2`, por lo que ambos pesos influyen en su inclinación. El bias `w0` determina principalmente el desplazamiento de la frontera.

## Fase 8. Evolución del error — Actividades 8.a y 8.b

In [ ]:
def plot_error(E_values,title):
    plt.figure(figsize=(7,4))
    plt.plot(range(1,len(E_values)+1),E_values,marker=".")
    plt.xlabel("Época"); plt.ylabel("Tasa de error")
    plt.title(title); plt.grid(alpha=.3); plt.show()
plot_error(E_values,"Evolución del error con lr = 0.5")
print("Error final:",E_values[-1])

**Pregunta 8.a.** El error no necesariamente disminuye de forma continua: una actualización que corrige un punto puede afectar la clasificación de otros, provocando aumentos o estancamientos temporales.

**Pregunta 8.b.** Error final cero significa que no hubo errores en la última época del entrenamiento. No garantiza que el modelo clasifique correctamente cualquier dato futuro o que generalice perfectamente.

## Fase 9. Comparación de tasas de aprendizaje — Actividades 9.a, 9.b y 9.c

In [ ]:
w_common=initialize_weights()
results={}
for lr in [0.3,0.5,0.8]:
    wf,ep,up,ws,es=perceptron_learning(X,y,w_common.copy(),lr,1000)
    results[lr]={"weights":wf,"epochs":ep,"updates":up,"W":ws,"E":es}
print("Pesos iniciales compartidos:",w_common)
print(f"{'lr':<7}{'Épocas':<12}{'Actualizaciones':<19}{'Error final':<15}Pesos finales")
for lr,r in results.items():
    print(f"{lr:<7}{r['epochs']:<12}{r['updates']:<19}{r['E'][-1]:<15.4f}{[round(v,4) for v in r['weights']]}")

In [ ]:
for lr,r in results.items():
    plot_six_states(X,y,r["W"],f"Frontera con lr = {lr}")
    plot_error(r["E"],f"Error con lr = {lr}")

In [ ]:
min_ep=min(r['epochs'] for r in results.values())
mejores=[lr for lr,r in results.items() if r['epochs']==min_ep]
print("Tasa(s) con menos épocas:",mejores,"| Épocas:",min_ep)
print("¿Todas convergieron?:",all(r['E'][-1]==0 for r in results.values()))

**Pregunta 9.a.** Usar diferentes pesos iniciales introduciría otra variable en el experimento: no se podría atribuir la diferencia de resultados solamente a la tasa de aprendizaje.

**Pregunta 9.b.** La celda anterior identifica qué tasa necesitó menos épocas. Eso no implica necesariamente que sea la mejor en todos los problemas: también importan la estabilidad, el número de actualizaciones y la generalización.

**Pregunta 9.c.** Los pesos finales pueden ser distintos. Varias rectas diferentes pueden separar correctamente un mismo dataset linealmente separable.

## Conclusión personal
En esta práctica comprobé cómo el perceptrón aprende a clasificar dos grupos de puntos ajustando sus pesos después de cada error. Las gráficas permiten observar que la recta cambia de posición e inclinación durante el entrenamiento. También entendí que la tasa de aprendizaje influye en el proceso y que obtener cero errores en los datos de entrenamiento no asegura predicciones perfectas en datos nuevos.

**Referencia:** Guía de Práctica 03, *El Perceptrón*, Universidad Politécnica Salesiana.